In [1]:
from pathlib import Path
import warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")

# --- modelo ---
ORDEN = (1, 0, 0)          # AR(1) sobre ln(spread)
TREND = "c"                # con constante: en statsmodels 'const' es la MEDIA de ln(spread)

# --- datos ---
HOJA = "Diarias"
FECHA_INICIO = "2004-01-01"
FECHA_FIN = "2026-08-31"   # excluye septiembre de 2026, que está incompleto
S = 12                     # periodos por año (estacionalidad mensual)

# --- pronóstico y salidas ---
H = 10                     # meses
PUNTO = "mediana"          # "mediana" = exp(μ) (consistente con los IC) | "media" = exp(μ + σ²/2)
VENTANA_GRAFICO = 300      # muestra completa desde 2004 y el horizonte hasta 2027
LAGS_LB = [6, 12, 18, 24] # rezagos del Ljung-Box
LAGS_ARCH = [1, 3, 6]     # rezagos del ARCH-LM
AUTORES = "Felipe Ramirez Beltra, Nicolas Gordillo Luna, Julian Mauricio Molina Perez"

# --- rutas ---
BASE_DIR = next((r for r in (Path.cwd(), *Path.cwd().parents)
                 if (r / "base de datos" / "Datos_EMBI.xlsx").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("No se encontró base de datos/Datos_EMBI.xlsx en la carpeta del proyecto ni en sus carpetas superiores.")
ARCHIVO = BASE_DIR / "base de datos" / "Datos_EMBI.xlsx"
SALIDA = BASE_DIR / "code" / "resultados_embi"
SALIDA.mkdir(parents=True, exist_ok=True)

# --- estilo gráfico claro (figuras y póster) ---
plt.style.use("default")
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
                     "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
C_AZUL, C_VERDE, C_ROJO = "#1F4E78", "#2E7D32", "#C0392B"
C_IC95, C_IC90 = "#F8BBD0", "#EC407A"

R = {}                     # resultados de cada etapa; el póster solo lee este diccionario


## 1. Datos: promedios mensuales, enero de 2004 a agosto de 2026

In [2]:
R = {}   # diccionario de resultados

def cargar_serie(archivo, hoja):
    """Lee fecha y valor (dos primeras columnas) y devuelve la serie indexada por fecha; omite filas inválidas."""
    raw = pd.read_excel(archivo, sheet_name=hoja, header=1)
    col_fecha, col_valor = raw.columns[:2]
    fecha = pd.to_datetime(raw[col_fecha], errors="coerce")
    valor = pd.to_numeric(raw[col_valor], errors="coerce")
    validas = fecha.notna() & valor.notna()
    if not validas.any():
        raise ValueError("No se encontraron filas con fecha y spread válidos.")
    if not validas.all():
        print(f"Filas omitidas por fecha o spread inválido/vacío: {(~validas).sum()}")
    return pd.Series(valor[validas].to_numpy(), index=pd.DatetimeIndex(fecha[validas]), name="spread").sort_index()

def a_mensual(serie, desde=FECHA_INICIO, hasta=FECHA_FIN):
    """Promedia las observaciones diarias de cada mes, dentro del intervalo indicado."""
    serie = serie.loc[(serie.index >= pd.Timestamp(desde)) & (serie.index <= pd.Timestamp(hasta))]
    if serie.empty:
        raise ValueError(f"No hay observaciones válidas entre {desde} y {hasta}.")
    mensual = serie.resample("MS").mean()
    esperados = pd.date_range(pd.Timestamp(desde).to_period("M").to_timestamp(),
                              pd.Timestamp(hasta).to_period("M").to_timestamp(), freq="MS")
    faltantes = esperados.difference(mensual.dropna().index)
    if len(faltantes):
        raise ValueError(f"Faltan observaciones para {len(faltantes)} mes(es): {faltantes.strftime('%m/%Y').tolist()}")
    return mensual.reindex(esperados).rename("spread")

diaria = cargar_serie(ARCHIVO, HOJA)
y = a_mensual(diaria)   # promedio mensual del spread (pbs), 01/2004–08/2026
if (y <= 0).any():
    raise ValueError("El logaritmo requiere promedios mensuales positivos.")
ly = np.log(y)          # ln(spread)

etq_agr = "promedio mensual de observaciones diarias"
R.update(n=len(y), ini=f"{y.index[0]:%m/%Y}", fin=f"{y.index[-1]:%m/%Y}", ultimo=float(y.iloc[-1]), agr=etq_agr)
print(f"Datos originales: {diaria.index.min():%d/%m/%Y} a {diaria.index.max():%d/%m/%Y} ({len(diaria)} filas)")
print(f"Promedios mensuales entre {FECHA_INICIO} y {FECHA_FIN}: {len(y)} obs., {y.index[0]:%m/%Y} a {y.index[-1]:%m/%Y}")
print(y.head(3))

Datos originales: 02/01/1998 a 03/09/2026 (7480 filas)
Promedios mensuales entre 2004-01-01 y 2026-08-31: 272 obs., 01/2004 a 08/2026
2004-01-01    389.000000
2004-02-01    433.700000
2004-03-01    402.304348
Freq: MS, Name: spread, dtype: float64


## 2. Transformación y varianza

In [3]:
def grafico_transformacion(y, ly, ruta, etq):
    fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(y.index, y, color=C_AZUL, lw=1.6)
    ax[0].set_title(f"ANTES: Spread EMBIG Colombia (pbs), {etq}"); ax[0].set_ylabel("pbs")
    ax[1].plot(ly.index, ly, color=C_VERDE, lw=1.6)
    ax[1].axhline(ly.mean(), color="gray", lw=.8, ls="--")
    ax[1].set_title(f"DESPUÉS: ln(spread), {etq} (serie del AR(1); línea = media)"); ax[1].set_ylabel("ln")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

def tabla_varianza(series, etq):
    """Levene entre mitades y ARCH(1), ARCH(4) sobre residuos de un AR(1) auxiliar."""
    def arch_ar1(s, lags):
        s = np.asarray(s)
        return het_arch(sm.OLS(s[1:], sm.add_constant(s[:-1])).fit().resid, nlags=lags)[1]
    print(f"Serie mensual: {etq}")
    for nombre, s in series.items():
        s = s.reset_index(drop=True); h = len(s) // 2
        print(f"{nombre:6s} var1={s[:h].var():.4f} var2={s[h:].var():.4f} "
              f"Levene p={stats.levene(s[:h], s[h:]).pvalue:.4f} | "
              f"ARCH(1) p={arch_ar1(s,1):.4f} ARCH(4) p={arch_ar1(s,4):.4f}")

grafico_transformacion(y, ly, SALIDA / "fig1_transformacion.png", etq_agr)
print("--- Diagnóstico de varianza ---")
tabla_varianza({"Nivel": y, "ln": ly}, etq_agr)


--- Diagnóstico de varianza ---
Serie mensual: promedio mensual de observaciones diarias
Nivel  var1=11250.2243 var2=5646.9104 Levene p=0.2717 | ARCH(1) p=0.6539 ARCH(4) p=0.9783
ln     var1=0.1564 var2=0.0769 Levene p=0.0209 | ARCH(1) p=0.6904 ARCH(4) p=0.9549


## 3. Etapa 1 — Identificación sobre ln(spread)

In [4]:
def prueba_adf(s, reg):
    r = adfuller(s, regression=reg, autolag="AIC")
    print(f"ADF  reg={reg:2s} est={r[0]:.3f} p={r[1]:.4f} rezagos={r[2]} crit5%={r[4]['5%']:.3f}")
    return r[1]

def prueba_kpss(s, reg):
    r = kpss(s, regression=reg, nlags=int(4 * (len(s) / 100) ** 0.25))
    print(f"KPSS reg={reg:2s} est={r[0]:.3f} p={r[1]:.4f} crit5%={r[3]['5%']}")
    return r[1]

def grafico_fac_facp(y, ly, ruta, lags=36):
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    for fila, (s, nom) in enumerate([(y, "spread original"), (ly, "ln(spread)")]):
        plot_acf(s, lags=lags, ax=ax[fila, 0], title=f"FAC – {nom}")
        plot_pacf(s, lags=lags, ax=ax[fila, 1], method="ywm", title=f"FACP – {nom}")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

print("=== ETAPA 1: IDENTIFICACIÓN (ln(spread)) ===")
adf_ly = prueba_adf(ly, "c"); kpss_ly = prueba_kpss(ly, "c")
adf_ct = prueba_adf(ly, "ct"); kpss_ct = prueba_kpss(ly, "ct")
grafico_fac_facp(y, ly, SALIDA / "fig2_FAC_FACP.png")
R["ident"] = dict(adf_ly=adf_ly, kpss_ly=kpss_ly, adf_ct=adf_ct, kpss_ct=kpss_ct)


=== ETAPA 1: IDENTIFICACIÓN (ln(spread)) ===
ADF  reg=c  est=-3.304 p=0.0147 rezagos=1 crit5%=-2.872
KPSS reg=c  est=0.620 p=0.0208 crit5%=0.463
ADF  reg=ct est=-3.422 p=0.0485 rezagos=1 crit5%=-3.427
KPSS reg=ct est=0.438 p=0.0100 crit5%=0.146


## 4. Etapa 2 — Estimación del AR(1)

In [5]:
def ajustar_ar1(ly):
    """AR(1) con constante por máxima verosimilitud. En statsmodels, 'const' es la media de ln(spread)."""
    m = ARIMA(ly, order=ORDEN, trend=TREND).fit()
    phi, mu = m.params["ar.L1"], m.params["const"]
    ic = m.conf_int().loc["ar.L1"].to_numpy()
    c = (1 - phi) * mu
    info = dict(phi=phi, se_phi=m.bse["ar.L1"], p_phi=m.pvalues["ar.L1"], ic_phi=ic, ic_incluye_1=bool(ic[1] >= 1),
                mu=mu, c=c, se_mu=m.bse["const"], p_mu=m.pvalues["const"], sigma2=m.params["sigma2"],
                vida_media=np.log(.5) / np.log(phi), nivel_lp=float(np.exp(mu)), aic=m.aic, bic=m.bic)
    return m, info

print("=== ETAPA 2: ESTIMACIÓN ===")
modelo, M = ajustar_ar1(ly)
print(modelo.summary().tables[1])
print(f"φ={M['phi']:.3f} (EE {M['se_phi']:.3f})  IC95%=[{M['ic_phi'][0]:.3f}, {M['ic_phi'][1]:.3f}]  (¿incluye 1? {'sí' if M['ic_incluye_1'] else 'no'})")
print(f"Media de largo plazo de ln = {M['mu']:.4f} (SE {M['se_mu']:.3f}) | sigma2 = {M['sigma2']:.4f}")
print(f"Intercepto AR(1): c=(1 - phi)*mu=(1 - {M['phi']:.4f})*{M['mu']:.4f}={M['c']:.4f}")
print(f"Vida media={M['vida_media']:.1f} meses | nivel de largo plazo={M['nivel_lp']:.0f} pbs | AIC={M['aic']:.2f} BIC={M['bic']:.2f}")
R["modelo"] = M


=== ETAPA 2: ESTIMACIÓN ===
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.4803      0.140     39.167      0.000       5.206       5.755
ar.L1          0.9509      0.019     49.376      0.000       0.913       0.989
sigma2         0.0122      0.001     24.290      0.000       0.011       0.013
φ=0.951 (EE 0.019)  IC95%=[0.913, 0.989]  (¿incluye 1? no)
Media de largo plazo de ln = 5.4803 (SE 0.140) | sigma2 = 0.0122
Intercepto AR(1): c=(1 - phi)*mu=(1 - 0.9509)*5.4803=0.2693
Vida media=13.8 meses | nivel de largo plazo=240 pbs | AIC=-417.85 BIC=-407.03


## 5. Etapa 3 — Diagnóstico de residuos

In [6]:
def diagnosticar(m):
    """Pruebas sobre los residuos: Ljung-Box, ARCH-LM, Jarque-Bera."""
    res = np.asarray(m.resid)
    p, _, q = m.model.order
    lb = acorr_ljungbox(res, lags=LAGS_LB, model_df=p + q)["lb_pvalue"].to_numpy()
    arch = np.array([het_arch(res, nlags=k)[1] for k in LAGS_ARCH])
    return dict(lb_p=lb, arch_p=arch, jb_p=stats.jarque_bera(res)[1],
                asim=stats.skew(res), curt=stats.kurtosis(res, fisher=False))

def grafico_residuos(m, ruta, lags=36):
    """Residuales, FAC, FAC de residuos² (volatilidad) y Q-Q (normalidad)."""
    res = pd.Series(np.asarray(m.resid), index=ly.index)
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    ax[0, 0].plot(res, color="#444", lw=1); ax[0, 0].axhline(0, color="gray", lw=.8); ax[0, 0].set_title("Residuales AR(1)")
    plot_acf(res, lags=lags, ax=ax[0, 1], title="FAC residuos")
    plot_acf(res ** 2, lags=lags, ax=ax[1, 0], title="FAC residuos² (volatilidad)")
    stats.probplot(res, dist="norm", plot=ax[1, 1]); ax[1, 1].set_title("Q-Q residuos (normalidad)")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

print("=== ETAPA 3: DIAGNÓSTICO ===")
D = diagnosticar(modelo)
print(f"Ljung-Box p {LAGS_LB}:", D["lb_p"].round(3).tolist())
print(f"ARCH-LM p {LAGS_ARCH}:", D["arch_p"].round(3).tolist())
print(f"Jarque-Bera p: {D['jb_p']:.2e} | asimetría={D['asim']:.2f} curtosis={D['curt']:.2f}")
grafico_residuos(modelo, SALIDA / "fig3_residuos.png")
R["diag"] = D


=== ETAPA 3: DIAGNÓSTICO ===
Ljung-Box p [6, 12, 18, 24]: [0.097, 0.298, 0.09, 0.124]
ARCH-LM p [1, 3, 6]: [0.705, 0.97, 0.988]
Jarque-Bera p: 1.99e-261 | asimetría=2.00 curtosis=12.48


## 6. Etapa 4 — Pronóstico (IC 95% y 90%, en pbs)

In [7]:
def pronosticar(modelo, fechas, h=H, punto="mediana"):
    """Mediana = exp(μ_h); media = exp(μ_h + σ²_h/2) bajo normalidad en logs. Los IC corresponden a la mediana."""
    fc = modelo.get_forecast(h)
    mu_h, var_h = fc.predicted_mean.values, np.asarray(fc.var_pred_mean)
    mediana, media = np.exp(mu_h), np.exp(mu_h + 0.5 * var_h)
    fut = [fechas[-1] + pd.DateOffset(months=i + 1) for i in range(h)]
    return dict(fc=fc, fut=fut, mediana=mediana, media=media, etq=punto,
                punto=mediana if punto == "mediana" else media,
                ic95=np.exp(fc.conf_int(alpha=0.05).values), ic90=np.exp(fc.conf_int(alpha=0.10).values))

def panel_pronostico(ax, y, P, a95, a90, color, titulo, ref95=None, nivel_lp=None):
    fut, punto = P["fut"], P["punto"]
    ax.plot(y.index[-VENTANA_GRAFICO:], y[-VENTANA_GRAFICO:], color=C_AZUL, lw=1.4, label="Observado")
    ax.plot([y.index[-1], *fut], [y.iloc[-1], *punto], "--o", ms=3, color=color, label=f"Pronóstico ({P['etq']})")
    ax.fill_between(fut, a95[:, 0], a95[:, 1], color=C_ROJO, alpha=.15, label="IC 95%")
    ax.fill_between(fut, a90[:, 0], a90[:, 1], color=C_AZUL, alpha=.25, label="IC 90%")
    if ref95 is not None:
        ax.plot(fut, ref95[:, 0], ":", color="gray", lw=1.3, label="IC 95% original"); ax.plot(fut, ref95[:, 1], ":", color="gray", lw=1.3)
    if nivel_lp is not None:
        ax.axhline(nivel_lp, color="gray", ls=":", lw=1, label="Nivel de largo plazo")
    ax.annotate(f"{punto[-1]:.0f}", xy=(fut[-1], punto[-1]), xytext=(0, 5), textcoords="offset points",
                ha="center", color=color, fontsize=8, fontweight="bold", zorder=10)
    ax.set_title(titulo, fontsize=10); ax.set_ylabel("pbs"); ax.legend(fontsize=8)

print("=== ETAPA 4: PRONÓSTICO ===")
P = pronosticar(modelo, y.index, punto=PUNTO)
pron = pd.DataFrame({"Fecha": [f"{f:%m/%Y}" for f in P["fut"]], "Mediana": P["mediana"], "Media": P["media"],
                     "IC95_inf": P["ic95"][:, 0], "IC95_sup": P["ic95"][:, 1],
                     "IC90_inf": P["ic90"][:, 0], "IC90_sup": P["ic90"][:, 1]})
print(pron.round(1).to_string(index=False))
print(f"Último dato ({y.index[-1]:%m/%Y}): {y.iloc[-1]:.0f} pbs | nivel de largo plazo: {M['nivel_lp']:.0f} pbs")
print(f"Se reporta la {PUNTO}. Los IC son los de la mediana.")
fig, ax = plt.subplots(figsize=(16, 5.2))
inicio_grafico = pd.Timestamp("2024-01-01")
panel_pronostico(ax, y, P, P["ic95"], P["ic90"], C_ROJO, "Historia y pronóstico (2024–2027)", nivel_lp=M["nivel_lp"])
datos_visibles = y.loc[inicio_grafico:]
y_min = min(float(datos_visibles.min()), float(P["ic95"][:, 0].min()))
y_max = max(float(datos_visibles.max()), float(P["ic95"][:, 1].max()))
margen = 0.05 * (y_max - y_min)
ax.set_xlim(inicio_grafico, P["fut"][-1] + pd.DateOffset(months=1))
ax.set_ylim(y_min - margen, y_max + margen)
plt.tight_layout(); fig.savefig(SALIDA / "fig4_pronostico.png", dpi=200); plt.show()
R["pron"] = P


=== ETAPA 4: PRONÓSTICO ===
  Fecha  Mediana  Media  IC95_inf  IC95_sup  IC90_inf  IC90_sup
09/2026    190.4  191.6     153.4     236.5     158.8     228.4
10/2026    192.6  194.9     142.8     259.7     149.9     247.6
11/2026    194.7  198.0     136.2     278.4     144.3     262.8
12/2026    196.7  200.9     131.5     294.4     140.3     275.9
01/2027    198.6  203.7     127.9     308.5     137.3     287.4
02/2027    200.5  206.4     125.1     321.2     135.0     297.8
03/2027    202.3  208.9     123.0     332.8     133.2     307.2
04/2027    204.0  211.3     121.2     343.3     131.8     315.7
05/2027    205.6  213.6     119.8     352.9     130.7     323.6
06/2027    207.2  215.7     118.6     361.8     129.8     330.8
Último dato (08/2026): 188 pbs | nivel de largo plazo: 240 pbs
Se reporta la mediana. Los IC son los de la mediana.


## 7. Etapa 6 — Póster (PDF, paleta clara)

In [8]:
# Paleta clara
BG, CARD, BORDE, PILL = "#F4F7FB", "#FFFFFF", "#D5DCEB", "#CFE3FF"
INK, MUTED, ACC, OK, KO, HEAD = "#1B2A4A", "#5A6785", "#D97706", "#1E9E5A", "#D64545", "#DCE8FA"

def textos_poster(R):
    """Frases del póster, construidas con las cifras y p-valores de R (cambian solas si cambian los datos)."""
    I, M, D, P = R["ident"], R["modelo"], R["diag"], R["pron"]
    raiz = ("se rechaza raíz unitaria" if I["adf_ly"] < 0.05 else
            "no se rechaza raíz unitaria; el test tiene poca potencia con φ cercano a 1")
    ident = (f"Serie mensual de enero de 2004 a agosto de 2026 ({R['agr']}). El spread en niveles tiene varianza cambiante, por lo que se usa ln. "
             f"ADF sobre ln(spread): p = {I['adf_ly']:.3f} ({raiz}); KPSS: p = {I['kpss_ly']:.3f}. "
             f"La FACP de ln(spread) tiene un pico dominante en el rezago 1 (patrón AR(1)), así que se modela en niveles de ln (d = 0). "
             f"φ = {M['phi']:.3f}, IC 95% [{M['ic_phi'][0]:.2f}, {M['ic_phi'][1]:.2f}] {'incluye' if M['ic_incluye_1'] else 'no incluye'} el 1.")
    interp_adf = ("se rechaza raíz unitaria; evidencia de estacionariedad en ln(spread)" if I["adf_ly"] < 0.05
                  else "no se rechaza raíz unitaria en ln(spread)")
    interp_kpss = ("se rechaza estacionariedad en ln(spread)" if I["kpss_ly"] < 0.05
                   else "no se rechaza estacionariedad en ln(spread)")
    if I["adf_ly"] < 0.05 and I["kpss_ly"] < 0.05:
        acuerdo = "Evidencia mixta: ambos rechazan su H₀; conclusión no concluyente."
    elif I["adf_ly"] < 0.05:
        acuerdo = "Ambos tests respaldan estacionariedad al 5%."
    elif I["kpss_ly"] < 0.05:
        acuerdo = "Evidencia de no estacionariedad al 5%."
    else:
        acuerdo = "Ningún test rechaza su H₀; evidencia no concluyente."
    modelo = (f"Vida media: {M['vida_media']:.1f} meses. Nivel de largo plazo: {M['nivel_lp']:.0f} pbs. "
              f"ADF (H₀: raíz unitaria), p={I['adf_ly']:.3f}: {interp_adf}. "
              f"KPSS (H₀: estacionariedad), p={I['kpss_ly']:.3f}: {interp_kpss}. {acuerdo}")
    sup = ("Los IC originales asumen errores normales; la normalidad se rechaza, por lo que deben interpretarse con cautela."
           if D["jb_p"] < 0.05 else "Los residuos son compatibles con errores normales.")
    pron = (f"Último dato: {R['ultimo']:.0f} pbs ({R['fin']}); pronóstico a {len(P['fut'])} meses hacia el nivel de largo plazo. "
            "Se muestran los IC 95% originales.")
    return dict(ident=ident, modelo=modelo, sup=sup, pron=pron)

def armar_poster(R, carpeta, salida_pdf, salida_png=None):
    T = textos_poster(R)
    M, D, P = R["modelo"], R["diag"], R["pron"]
    fig = plt.figure(figsize=(16.5, 23.4), facecolor=BG)
    bg = fig.add_axes([0, 0, 1, 1]); bg.set_xlim(0, 1); bg.set_ylim(0, 1); bg.axis("off")

    def card(x0, y0, w, h, titulo):
        bg.add_patch(FancyBboxPatch((x0, y0), w, h, boxstyle="round,pad=0,rounding_size=0.012", fc=CARD, ec=BORDE, lw=1.5))
        bg.text(x0 + 0.02, y0 + h - 0.006, titulo, color=INK, fontsize=22, fontweight="bold", va="center",
                bbox=dict(boxstyle="round,pad=0.5", fc=PILL, ec="none"))
    def texto(x, yv, s, w=60, size=15, color=INK, **kw):
        bg.text(x, yv, textwrap.fill(s, w), color=color, fontsize=size, va="top", ha="left", **kw)
    def imagen(rect, nombre):
        ax = fig.add_axes(rect); ax.imshow(plt.imread(carpeta / nombre)); ax.axis("off")

    # encabezado
    bg.add_patch(Rectangle((0, 0.915), 1, 0.085, fc=HEAD, ec="none"))
    bg.text(0.5, 0.968, "SPREAD EMBIG DE COLOMBIA", color=INK, fontsize=44, fontweight="bold", ha="center", va="center")
    bg.text(0.5, 0.936, f"Modelo AR(1) sobre ln(spread) | serie mensual {R['ini']} – {R['fin']} ({R['n']} obs.)",
            color=ACC, fontsize=20, ha="center", va="center", fontweight="bold")
    bg.text(0.03, 0.985, AUTORES, color=MUTED, fontsize=13, ha="left", va="center")

    # 1. serie original e identificación
    card(0.02, 0.655, 0.96, 0.250, "Serie original e identificación")
    imagen([0.03, 0.746, 0.34, 0.125], "fig1_transformacion.png")
    imagen([0.39, 0.746, 0.58, 0.125], "fig2_FAC_FACP.png")
    texto(0.03, 0.724, T["ident"], w=175, size=12)

    # 2. estimación del modelo
    card(0.02, 0.490, 0.96, 0.155, "Estimación del modelo AR(1)")
    datos_resumen = modelo.summary().tables[1].data
    ax_tabla = fig.add_axes([0.03, 0.515, 0.56, 0.105]); ax_tabla.axis("off")
    tabla = ax_tabla.table(cellText=datos_resumen[1:], colLabels=datos_resumen[0],
                           cellLoc="center", bbox=[0, 0, 1, 1])
    tabla.auto_set_font_size(False); tabla.set_fontsize(8)
    for (fila, columna), celda in tabla.get_celld().items():
        celda.set_edgecolor(BORDE); celda.set_linewidth(0.6)
        if fila == 0:
            celda.set_facecolor(PILL); celda.set_text_props(color=INK, weight="bold")
        else:
            celda.set_facecolor(CARD if fila % 2 else BG); celda.set_text_props(color=INK)
    texto(0.62, 0.598, T["modelo"], w=50, size=12)

    # 3. verificación de supuestos
    card(0.02, 0.270, 0.96, 0.210, "Verificación de supuestos")
    tests = [("Sin autocorrelación", f"Ljung-Box {LAGS_LB}: p = {', '.join(f'{v:.2f}' for v in D['lb_p'])}", D["lb_p"].min() > 0.05),
             ("Homocedasticidad", f"ARCH-LM {LAGS_ARCH}: p = {', '.join(f'{v:.2f}' for v in D['arch_p'])}", D["arch_p"].min() > 0.05),
             ("Normalidad", f"Jarque-Bera: p = {D['jb_p']:.1e}", D["jb_p"] > 0.05)]
    for i, (nom, det, ok) in enumerate(tests):
        yy = 0.430 - i * 0.038
        bg.text(0.03, yy, "✓" if ok else "✗", color=OK if ok else KO, fontsize=26, fontweight="bold", va="center")
        bg.text(0.055, yy + 0.008, nom, color=INK, fontsize=16, fontweight="bold", va="center")
        bg.text(0.055, yy - 0.010, det, color=MUTED, fontsize=13, va="center")
    texto(0.03, 0.322, T["sup"], w=62, size=13, color=MUTED)
    imagen([0.50, 0.280, 0.47, 0.185], "fig3_residuos.png")

    # 4. pronóstico completo
    card(0.02, 0.090, 0.96, 0.170, f"Pronóstico a {len(P['fut'])} meses (pbs)")
    et = P["etq"].capitalize()
    bg.text(0.03, 0.220, f"{'Fecha':10s}{et:>9s}{'IC95':>16s}", color=ACC, fontsize=9, family="monospace", va="top", fontweight="bold")
    for j, i in enumerate(range(len(P["fut"]))):
        a, b = P["ic95"][i, 0], P["ic95"][i, 1]
        bg.text(0.03, 0.208 - j * 0.0085, f"{P['fut'][i]:%m/%Y}  {P['punto'][i]:8.0f}  {f'[{a:.0f}, {b:.0f}]':>15s}",
                color=INK, fontsize=9, family="monospace", va="top")
    texto(0.03, 0.118, T["pron"], w=115, size=8, color=MUTED)
    imagen([0.48, 0.093, 0.49, 0.155], "fig4_pronostico.png")

    # 5. resultados
    card(0.02, 0.012, 0.96, 0.068, "Resultados")
    bg.text(0.5, 0.030, rf"$\ln S_t={M['c']:.3f}+{M['phi']:.3f}\,\ln S_{{t-1}}+\varepsilon_t$",
            color=INK, fontsize=26, ha="center", va="center")
    bg.text(0.98, 0.003, "Datos: Banco de la República (PD04715XD)", color=MUTED, fontsize=10, ha="right")

    fig.savefig(salida_pdf, facecolor=BG)
    if salida_png: fig.savefig(salida_png, dpi=60, facecolor=BG)
    plt.show()

armar_poster(R, SALIDA, SALIDA / "poster_embig.pdf", SALIDA / "poster_embig_preview.png")
